In [16]:
import customtkinter as ctk
import cv2
from PIL import Image
import requests
import base64
import threading
import os
from tkinter import filedialog

class AICore:
    def __init__(self):
        self.url = "http://localhost:11434/api/generate"
        self.model = "moondream:latest"
        self.lang = "English"
        # Keyboards with Matras
        self.keyboards = {
            "English": "QWERTYUIOPASDFGHJKLZXCVBNM,. ",
            "Hindi": "ािीुूृेैोौ्ंः अआइईउऊऋएऐओऔ कखगघङचछजझञटठडढणतथदधनपफबभमयरलवशषसह ",
            "Gujarati": "ાિીુૂૃેૈોૌ્ંઃ અઆઇઈઉઊઋએઐઓઔ કખગઘઙચછજઝઞટઠડઢણતથદધનપફબભમયરલવશષસહ "
        }

    def run_analysis(self, img_path, query, callback):
        def _task():
            try:
                with open(img_path, "rb") as f:
                    b64 = base64.b64encode(f.read()).decode('utf-8')
                
                prompt = (f"Analyze this crop in {self.lang}. Symptoms: {query}. "
                          "Strictly give Disease Name and 3 steps for Cure. No citations.")

                payload = {"model": self.model, "prompt": prompt, "images": [b64], "stream": False}
                response = requests.post(self.url, json=payload, timeout=90)
                
                if response.status_code == 200:
                    ans = response.json().get('response', "AI returned no text.")
                    callback(ans)
                else:
                    callback(f"Server Error: {response.status_code}")
            except Exception as e:
                callback(f"Connection Error: Ensure Ollama is running.")
        
        threading.Thread(target=_task).start()

class App(ctk.CTk):
    def __init__(self):
        super().__init__()
        self.ai = AICore()
        self.cap = None
        self.img_path = None
        self.setup_lang_select()

    def setup_lang_select(self):
        self.clear_screen()
        self.geometry("400x400")
        ctk.CTkLabel(self, text="Choose Language\nભાષા પસંદ કરો / भाषा चुनें", font=("Arial", 20)).pack(pady=40)
        for l in ["English", "Hindi", "Gujarati"]:
            ctk.CTkButton(self, text=l, command=lambda lang=l: self.launch_main(lang)).pack(pady=10)

    def launch_main(self, lang):
        self.ai.lang = lang
        self.clear_screen()
        self.geometry("1100x950")
        self.setup_main_ui()

    def setup_main_ui(self):
        ui = {"English": "Diagnosis Result", "Hindi": "निदान का परिणाम", "Gujarati": "નિદાનનું પરિણામ"}[self.ai.lang]
        
        self.status = ctk.CTkLabel(self, text="● SYSTEM READY", text_color="cyan", font=("Arial", 14, "bold"))
        self.status.pack(pady=5)

        self.display = ctk.CTkLabel(self, text="...", width=700, height=400, fg_color="black", corner_radius=15)
        self.display.pack(pady=10)

        self.entry = ctk.CTkEntry(self, placeholder_text="Enter details...", width=700)
        self.entry.pack(pady=5)

        # Matra Keyboard
        self.kb = ctk.CTkFrame(self)
        self.kb.pack(pady=10)
        chars = self.ai.keyboards[self.ai.lang]
        r, c = 0, 0
        for char in chars:
            ctk.CTkButton(self.kb, text=char, width=38, height=38, 
                          command=lambda ch=char: self.entry.insert("end", ch)).grid(row=r, column=c, padx=1, pady=1)
            c += 1
            if c > 15: r += 1; c = 0

        btn_f = ctk.CTkFrame(self, fg_color="transparent")
        btn_f.pack(pady=10)
        ctk.CTkButton(btn_f, text="📹 Cam", command=self.start_cam).grid(row=0, column=0, padx=5)
        ctk.CTkButton(btn_f, text="📁 Upload File", command=self.upload_file).grid(row=0, column=1, padx=5)
        ctk.CTkButton(btn_f, text="🎯 Diagnose", fg_color="green", command=self.diagnose).grid(row=0, column=2, padx=5)
        ctk.CTkButton(btn_f, text="🔄 Reset", fg_color="red", command=self.reset).grid(row=0, column=3, padx=5)

        ctk.CTkLabel(self, text=ui, font=("Arial", 14, "bold")).pack()
        self.output = ctk.CTkTextbox(self, width=900, height=180, font=("Helvetica", 15))
        self.output.pack(pady=10)

    def upload_file(self):
        file_path = filedialog.askopenfilename(filetypes=[("Image Files", "*.png *.jpg *.jpeg")])
        if file_path:
            self.img_path = file_path
            # Show the uploaded image in the preview box
            img = ctk.CTkImage(Image.open(file_path), size=(700, 400))
            self.display.configure(image=img, text="")
            self.status.configure(text="✅ FILE UPLOADED SUCCESSFULLY", text_color="#2ecc71")

    def start_cam(self):
        if not self.cap: self.cap = cv2.VideoCapture(0)
        self.status.configure(text="● CAMERA ACTIVE", text_color="cyan")
        self.update_stream()

    def update_stream(self):
        if self.cap:
            ret, frame = self.cap.read()
            if ret:
                img = ctk.CTkImage(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)), size=(700, 400))
                self.display.configure(image=img, text="")
                self.after(10, self.update_stream)

    def diagnose(self):
        # 1. Capture if camera is on, otherwise use the uploaded file
        if self.cap and not self.img_path:
            ret, frame = self.cap.read()
            if ret:
                self.img_path = "snapshot.jpg"
                cv2.imwrite(self.img_path, frame)
        
        if self.img_path:
            self.output.delete("1.0", "end")
            self.output.insert("1.0", "✅ IMAGE CONFIRMED. ANALYZING PATHOLOGY... PLEASE WAIT.")
            self.status.configure(text="● PROCESSING...", text_color="orange")
            self.ai.run_analysis(self.img_path, self.entry.get(), self.display_final_text)
        else:
            self.output.insert("1.0", "Error: Please upload a file or start the camera first.")

    def display_final_text(self, result):
        self.output.delete("1.0", "end")
        self.output.insert("1.0", result)
        self.status.configure(text="● ANALYSIS COMPLETE", text_color="#2ecc71")

    def reset(self):
        self.img_path = None
        if self.cap:
            self.cap.release()
            self.cap = None
        self.output.delete("1.0", "end")
        self.entry.delete(0, 'end')
        self.display.configure(image=None, text="System Reset")
        self.status.configure(text="● READY", text_color="cyan")

    def clear_screen(self):
        for w in self.winfo_children(): w.destroy()

if __name__ == "__main__":
    App().mainloop()